# Stage 8 — One-Time Locked Holdout Evaluation

**Project:** Heart_Attack_Risk_Assessment  
**Team:** team05 | **Student:** s502

This notebook performs the one-time final evaluation of the frozen Stage 6 XGBoost model on the untouched locked holdout.

Rules:
- no tuning;
- no retraining;
- no threshold search;
- no feature/preprocessing changes;
- frozen threshold reused exactly;
- locked holdout opened once;
- final metrics, fairness checks and confidence intervals saved;
- results logged to MLflow.

## 1. Install packages

In [1]:
%pip install -q -U mlflow sagemaker-mlflow scikit-learn xgboost joblib psutil
print("Packages ready.")

Note: you may need to restart the kernel to use updated packages.
Packages ready.


## 2. Paths and memory helper

In [2]:
from pathlib import Path
import json, os, gc
import boto3, joblib, psutil
import numpy as np
import pandas as pd

EXPECTED_PROJECT_FOLDER = "Heart_Attack_Risk_Assessment"
current = Path.cwd().resolve()

if current.name == EXPECTED_PROJECT_FOLDER:
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = next((p for p in current.parents if p.name == EXPECTED_PROJECT_FOLDER), None)

if PROJECT_ROOT is None:
    fallback = Path("/home/sagemaker-user/Heart_Attack_Risk_Assessment")
    if fallback.exists():
        PROJECT_ROOT = fallback
    else:
        raise FileNotFoundError("Cannot locate project root.")

DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "config"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
STAGE6_DIR = ARTIFACT_DIR / "stage6"
STAGE7_DIR = ARTIFACT_DIR / "stage7"
STAGE8_DIR = ARTIFACT_DIR / "stage8"

METRICS_DIR = STAGE8_DIR / "metrics"
FAIRNESS_DIR = STAGE8_DIR / "fairness"
GOVERNANCE_DIR = STAGE8_DIR / "governance"
PREDICTION_DIR = STAGE8_DIR / "predictions"

for d in [METRICS_DIR, FAIRNESS_DIR, GOVERNANCE_DIR, PREDICTION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def memory_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

print("Project root:", PROJECT_ROOT)
print("Stage 8 dir :", STAGE8_DIR)
print(f"Memory      : {memory_mb():.1f} MB")

Project root: /home/sagemaker-user/Heart_Attack_Risk_Assessment
Stage 8 dir : /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage8
Memory      : 161.7 MB


## 3. Validate required files

In [4]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/home/sagemaker-user/Heart_Attack_Risk_Assessment"
)

print("=" * 80)
print("CSV FILES IN PROJECT")
print("=" * 80)

csv_files = sorted(
    PROJECT_ROOT.rglob("*.csv")
)

for p in csv_files:
    try:
        size_mb = p.stat().st_size / (1024 ** 2)
        print(
            f"{size_mb:8.2f} MB | "
            f"{p.relative_to(PROJECT_ROOT)}"
        )
    except Exception:
        print(p)

print("\n" + "=" * 80)
print("FILES CONTAINING holdout/test/split")
print("=" * 80)

keywords = [
    "holdout",
    "test",
    "split",
    "locked",
    "validation"
]

for p in PROJECT_ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()

        if any(k in name for k in keywords):
            print(p.relative_to(PROJECT_ROOT))

CSV FILES IN PROJECT
    7.31 MB | artifacts/stage6/oof/logistic_regression_oof_predictions.csv
    7.24 MB | artifacts/stage6/oof/random_forest_oof_predictions.csv
    4.34 MB | artifacts/stage6/oof/xgboost_oof_predictions.csv
    0.00 MB | artifacts/stage6/tuning/logistic_regression_tuning_results.csv
    0.00 MB | artifacts/stage6/tuning/random_forest_tuning_results.csv
    0.00 MB | artifacts/stage6/tuning/xgboost_tuning_results.csv
    0.00 MB | artifacts/stage7/errors/error_counts.csv
    0.06 MB | artifacts/stage7/errors/false_negative_review_sample.csv
    0.07 MB | artifacts/stage7/errors/false_positive_review_sample.csv
    0.00 MB | artifacts/stage7/fairness/fairness_age.csv
    0.00 MB | artifacts/stage7/fairness/fairness_governance_summary.csv
    0.00 MB | artifacts/stage7/fairness/fairness_race.csv
    0.00 MB | artifacts/stage7/fairness/fairness_sex.csv
    0.01 MB | artifacts/stage7/shap/.ipynb_checkpoints/global_shap_importance-checkpoint.csv
    0.00 MB | artifacts/s

In [5]:
MLFLOW_CONFIG_FILE = PROJECT_ROOT / "mlflow_app_config_team05_s502.json"
FEATURE_METADATA_FILE = CONFIG_DIR / "feature_metadata.json"
STAGE6_NOMINATION_FILE = CONFIG_DIR / "stage6_candidate_nomination.json"
STAGE7_GOVERNANCE_FILE = STAGE7_DIR / "governance" / "stage7_governance_summary.json"
FROZEN_MODEL_FILE = STAGE6_DIR / "models" / "xgboost_best_estimator.joblib"
LOCKED_HOLDOUT_FILE = DATA_DIR / "full_locked_holdout_raw.csv"

required = {
    "MLflow config": MLFLOW_CONFIG_FILE,
    "Feature metadata": FEATURE_METADATA_FILE,
    "Stage 6 nomination": STAGE6_NOMINATION_FILE,
    "Stage 7 governance": STAGE7_GOVERNANCE_FILE,
    "Frozen XGBoost model": FROZEN_MODEL_FILE,
    "Locked holdout": LOCKED_HOLDOUT_FILE,
}

missing=[]
for label,path in required.items():
    status="FOUND" if path.exists() else "MISSING"
    print(f"{status:7} | {label:28} | {path}")
    if not path.exists():
        missing.append(str(path))

if missing:
    raise FileNotFoundError("Stage 8 missing files:\n" + "\n".join(missing))

print("[OK] Stage 8 inputs ready.")

FOUND   | MLflow config                | /home/sagemaker-user/Heart_Attack_Risk_Assessment/mlflow_app_config_team05_s502.json
FOUND   | Feature metadata             | /home/sagemaker-user/Heart_Attack_Risk_Assessment/config/feature_metadata.json
FOUND   | Stage 6 nomination           | /home/sagemaker-user/Heart_Attack_Risk_Assessment/config/stage6_candidate_nomination.json
FOUND   | Stage 7 governance           | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage7/governance/stage7_governance_summary.json
FOUND   | Frozen XGBoost model         | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage6/models/xgboost_best_estimator.joblib
FOUND   | Locked holdout               | /home/sagemaker-user/Heart_Attack_Risk_Assessment/data/full_locked_holdout_raw.csv
[OK] Stage 8 inputs ready.


## 4. Load frozen candidate and Stage 7 governance

In [6]:
with open(STAGE6_NOMINATION_FILE, "r", encoding="utf-8") as f:
    stage6_nomination = json.load(f)

FROZEN_MODEL_NAME = stage6_nomination["leading_candidate"]
FROZEN_THRESHOLD = float(stage6_nomination["selected_threshold"])

with open(STAGE7_GOVERNANCE_FILE, "r", encoding="utf-8") as f:
    stage7_governance = json.load(f)

print("Frozen model     :", FROZEN_MODEL_NAME)
print("Frozen threshold :", FROZEN_THRESHOLD)
print("Stage 7 status   :", stage7_governance.get("stage7_status"))
print("Stage 7 triggers :", stage7_governance.get("fairness_triggered_dimensions"))

if FROZEN_MODEL_NAME != "xgboost":
    raise ValueError("This Stage 8 notebook expects XGBoost as the frozen candidate.")

Frozen model     : xgboost
Frozen threshold : 0.52
Stage 7 status   : REVIEW REQUIRED
Stage 7 triggers : ['age', 'sex']


## 5. Connect to Team05 MLflow

In [7]:
import mlflow

with open(MLFLOW_CONFIG_FILE, "r", encoding="utf-8") as f:
    cfg = json.load(f)

REGION = cfg["REGION"]
MLFLOW_APP_ARN = cfg["MLFLOW_APP_ARN"]
MLFLOW_EXPERIMENT = cfg["EXPERIMENT_NAME"]
TEAM_ID = cfg["TEAM_ID"]
STUDENT_ID = cfg["STUDENT_ID"]
PROJECT_NAME = cfg["PROJECT_NAME"]

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print("MLflow experiment:", MLFLOW_EXPERIMENT)

MLflow experiment: ITI113/team05/Experiment1


## 6. Open the locked holdout once

In [8]:
with open(FEATURE_METADATA_FILE, "r", encoding="utf-8") as f:
    FEATURE_METADATA = json.load(f)

FULL_FEATURES = FEATURE_METADATA["full"]["features"]
TARGET = "HadHeartAttack"

needed_columns = list(dict.fromkeys(FULL_FEATURES + [TARGET]))

locked_holdout = pd.read_csv(
    LOCKED_HOLDOUT_FILE,
    usecols=needed_columns
)

frozen_pipeline = joblib.load(FROZEN_MODEL_FILE)

X_holdout = locked_holdout[FULL_FEATURES]
y_holdout = locked_holdout[TARGET].astype(np.int8).to_numpy()

print("Locked holdout rows:", len(locked_holdout))
print("Positive prevalence:", f"{y_holdout.mean():.4%}")
print(f"Memory after load    : {memory_mb():.1f} MB")
print("\n*** LOCKED HOLDOUT OPENED FOR FINAL EVALUATION ***")

Locked holdout rows: 88414
Positive prevalence: 5.6801%
Memory after load    : 357.6 MB

*** LOCKED HOLDOUT OPENED FOR FINAL EVALUATION ***


## 7. Generate final frozen-model predictions

In [9]:
holdout_scores = frozen_pipeline.predict_proba(X_holdout)[:,1]
holdout_predictions = (holdout_scores >= FROZEN_THRESHOLD).astype(np.int8)

print("Threshold used:", FROZEN_THRESHOLD)
print("Predicted positives:", int(holdout_predictions.sum()))

prediction_file = PREDICTION_DIR / "locked_holdout_predictions.csv"

pd.DataFrame({
    "actual": y_holdout,
    "association_score": holdout_scores,
    "prediction": holdout_predictions
}).to_csv(prediction_file, index=False)

print("Saved:", prediction_file)

Threshold used: 0.52
Predicted positives: 20739
Saved: /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage8/predictions/locked_holdout_predictions.csv


## 8. Final holdout metrics

In [10]:
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)

tn,fp,fn,tp = confusion_matrix(
    y_holdout,
    holdout_predictions,
    labels=[0,1]
).ravel()

final_metrics = {
    "accuracy": float(accuracy_score(y_holdout, holdout_predictions)),
    "recall": float(recall_score(y_holdout, holdout_predictions, zero_division=0)),
    "precision": float(precision_score(y_holdout, holdout_predictions, zero_division=0)),
    "f1": float(f1_score(y_holdout, holdout_predictions, zero_division=0)),
    "pr_auc": float(average_precision_score(y_holdout, holdout_scores)),
    "roc_auc": float(roc_auc_score(y_holdout, holdout_scores)),
    "brier_score": float(brier_score_loss(y_holdout, holdout_scores)),
    "specificity": float(tn/(tn+fp)),
    "fpr": float(fp/(fp+tn)),
    "fnr": float(fn/(fn+tp)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "threshold": float(FROZEN_THRESHOLD),
    "holdout_rows": int(len(y_holdout)),
    "positive_prevalence": float(y_holdout.mean()),
}

print(json.dumps(final_metrics, indent=2))

(METRICS_DIR/"locked_holdout_final_metrics.json").write_text(
    json.dumps(final_metrics, indent=2),
    encoding="utf-8"
)

{
  "accuracy": 0.798730970208338,
  "recall": 0.7931103146156909,
  "precision": 0.19205361878586238,
  "f1": 0.3092271262761539,
  "pr_auc": 0.4198124691659367,
  "roc_auc": 0.8846650440765925,
  "brier_score": 0.1491195112466812,
  "specificity": 0.799069455103607,
  "fpr": 0.20093054489639295,
  "fnr": 0.20688968538430905,
  "tn": 66636,
  "fp": 16756,
  "fn": 1039,
  "tp": 3983,
  "threshold": 0.52,
  "holdout_rows": 88414,
  "positive_prevalence": 0.056800959124120615
}


480

## 9. Confusion matrix

In [11]:
confusion_table = pd.DataFrame(
    [[tn,fp],[fn,tp]],
    index=["Actual Negative","Actual Positive"],
    columns=["Predicted Negative","Predicted Positive"]
)

display(confusion_table)
confusion_table.to_csv(METRICS_DIR/"locked_holdout_confusion_matrix.csv")

,Predicted Negative,Predicted Positive
Actual Negative,66636,16756
Actual Positive,1039,3983


## 10. Bootstrap 95% confidence intervals

In [12]:
BOOTSTRAP_REPETITIONS = 500
rng = np.random.default_rng(42)
n = len(y_holdout)

rows=[]

for _ in range(BOOTSTRAP_REPETITIONS):
    idx = rng.integers(0,n,size=n)
    yt = y_holdout[idx]
    ys = holdout_scores[idx]
    yp = holdout_predictions[idx]

    if len(np.unique(yt)) < 2:
        continue

    rows.append({
        "recall": recall_score(yt,yp,zero_division=0),
        "precision": precision_score(yt,yp,zero_division=0),
        "f1": f1_score(yt,yp,zero_division=0),
        "pr_auc": average_precision_score(yt,ys),
        "roc_auc": roc_auc_score(yt,ys),
        "brier_score": brier_score_loss(yt,ys),
    })

bootstrap_df = pd.DataFrame(rows)

ci_rows=[]
for metric in ["recall","precision","f1","pr_auc","roc_auc","brier_score"]:
    ci_rows.append({
        "metric":metric,
        "estimate":final_metrics[metric],
        "ci_2_5":float(bootstrap_df[metric].quantile(0.025)),
        "ci_97_5":float(bootstrap_df[metric].quantile(0.975)),
    })

confidence_intervals = pd.DataFrame(ci_rows)
display(confidence_intervals.round(4))

confidence_intervals.to_csv(
    METRICS_DIR/"locked_holdout_confidence_intervals.csv",
    index=False
)

,metric,estimate,ci_2_5,ci_97_5
0,recall,0.7931,0.7816,0.8044
1,precision,0.1921,0.1867,0.1976
2,f1,0.3092,0.3017,0.3165
3,pr_auc,0.4198,0.4041,0.4355
4,roc_auc,0.8847,0.8798,0.8889
5,brier_score,0.1491,0.1477,0.1505


## 11. Detect demographic subgroup columns

In [13]:
def first_existing(df,candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

AGE_COLUMN = first_existing(locked_holdout, ["AgeCategory","Age","AgeGroup"])
SEX_COLUMN = first_existing(locked_holdout, ["Sex","Gender"])
RACE_COLUMN = first_existing(
    locked_holdout,
    ["RaceEthnicityCategory","RaceEthnicity","Race"]
)

print("Age :", AGE_COLUMN)
print("Sex :", SEX_COLUMN)
print("Race:", RACE_COLUMN)

Age : AgeCategory
Sex : Sex
Race: RaceEthnicityCategory


## 12. Holdout fairness analysis

In [14]:
fairness_df = locked_holdout[
    [c for c in [AGE_COLUMN,SEX_COLUMN,RACE_COLUMN] if c is not None]
].copy()

fairness_df["_actual"] = y_holdout
fairness_df["_prediction"] = holdout_predictions

def subgroup_metrics(df, group_col):
    rows=[]

    for group,sub in df.groupby(group_col,dropna=False):
        yt=sub["_actual"].to_numpy(dtype=np.int8)
        yp=sub["_prediction"].to_numpy(dtype=np.int8)

        tn_g,fp_g,fn_g,tp_g = confusion_matrix(
            yt,yp,labels=[0,1]
        ).ravel()

        rows.append({
            "group":str(group),
            "n":int(len(sub)),
            "positive_actual_n":int((yt==1).sum()),
            "recall":float(recall_score(yt,yp,zero_division=0)),
            "precision":float(precision_score(yt,yp,zero_division=0)),
            "fnr":float(fn_g/(fn_g+tp_g)) if (fn_g+tp_g) else np.nan,
            "fpr":float(fp_g/(fp_g+tn_g)) if (fp_g+tn_g) else np.nan,
            "tn":int(tn_g),"fp":int(fp_g),"fn":int(fn_g),"tp":int(tp_g),
        })

    return pd.DataFrame(rows)

holdout_fairness_tables={}

for label,column in {"age":AGE_COLUMN,"sex":SEX_COLUMN,"race":RACE_COLUMN}.items():
    if column is None:
        continue

    table=subgroup_metrics(fairness_df,column)
    holdout_fairness_tables[label]=table

    print("\n",label.upper())
    display(table.round(4))

    table.to_csv(
        FAIRNESS_DIR/f"holdout_fairness_{label}.csv",
        index=False
    )


 AGE


,group,n,positive_actual_n,recall,precision,fnr,fpr,tn,fp,fn,tp
0,Age 18 to 24,5401,23,0.3043,0.2121,0.6957,0.0048,5352,26,16,7
1,Age 25 to 29,4392,26,0.2692,0.2188,0.7308,0.0057,4341,25,19,7
2,Age 30 to 34,5130,41,0.2927,0.1446,0.7073,0.0140,5018,71,29,12
3,Age 35 to 39,5669,53,0.3396,0.1224,0.6604,0.0230,5487,129,35,18
4,Age 40 to 44,6002,91,0.3187,0.1330,0.6813,0.0320,5722,189,62,29
5,Age 45 to 49,5645,124,0.5968,0.1558,0.4032,0.0726,5120,401,50,74
6,Age 50 to 54,6663,230,0.6870,0.1829,0.3130,0.1097,5727,706,72,158
7,Age 55 to 59,7335,434,0.7535,0.2100,0.2465,0.1782,5671,1230,107,327
8,Age 60 to 64,8727,540,0.7685,0.1857,0.2315,0.2223,6367,1820,125,415
9,Age 65 to 69,9396,760,0.7724,0.2102,0.2276,0.2554,6430,2206,173,587



 SEX


,group,n,positive_actual_n,recall,precision,fnr,fpr,tn,fp,fn,tp
0,Female,46783,1895,0.7203,0.1665,0.2797,0.1522,38054,6834,530,1365
1,Male,41631,3127,0.8372,0.2088,0.1628,0.2577,28582,9922,509,2618



 RACE


,group,n,positive_actual_n,recall,precision,fnr,fpr,tn,fp,fn,tp
0,"Black only, Non-Hispanic",6879,314,0.7611,0.1566,0.2389,0.1960,5278,1287,75,239
1,Hispanic,8554,342,0.6725,0.1765,0.3275,0.1307,7139,1073,112,230
2,"Multiracial, Non-Hispanic",1912,112,0.8393,0.2260,0.1607,0.1789,1478,322,18,94
3,"Other race only, Non-Hispanic",4453,181,0.8343,0.1953,0.1657,0.1456,3650,622,30,151
4,"White only, Non-Hispanic",63912,3887,0.8032,0.1949,0.1968,0.2148,47132,12893,765,3122
5,nan,2704,186,0.7903,0.2082,0.2097,0.2220,1959,559,39,147


## 13. Apply the same 0.10 recall-gap governance trigger

In [15]:
FAIRNESS_RECALL_GAP_TRIGGER = 0.10
summary_rows=[]

for label,table in holdout_fairness_tables.items():
    usable=table[table["positive_actual_n"]>0].copy()

    if len(usable)<2:
        gap=np.nan
        status="INSUFFICIENT GROUP DATA"
    else:
        gap=float(usable["recall"].max()-usable["recall"].min())
        status="INVESTIGATE" if gap>FAIRNESS_RECALL_GAP_TRIGGER else "NO TRIGGER"

    summary_rows.append({
        "dimension":label,
        "recall_gap":gap,
        "trigger_threshold":FAIRNESS_RECALL_GAP_TRIGGER,
        "governance_trigger":status
    })

holdout_fairness_summary=pd.DataFrame(summary_rows)

display(holdout_fairness_summary.round(4))

holdout_fairness_summary.to_csv(
    FAIRNESS_DIR/"holdout_fairness_governance_summary.csv",
    index=False
)

,dimension,recall_gap,trigger_threshold,governance_trigger
0,age,0.6391,0.1,INVESTIGATE
1,sex,0.1169,0.1,INVESTIGATE
2,race,0.1668,0.1,INVESTIGATE


## 14. Compare Stage 6 OOF vs Stage 8 holdout

In [16]:
comparison = pd.DataFrame([
    {"metric":"recall","stage6_oof":stage6_nomination.get("recall"),"stage8_holdout":final_metrics["recall"]},
    {"metric":"precision","stage6_oof":stage6_nomination.get("precision"),"stage8_holdout":final_metrics["precision"]},
    {"metric":"f1","stage6_oof":stage6_nomination.get("f1"),"stage8_holdout":final_metrics["f1"]},
    {"metric":"pr_auc","stage6_oof":stage6_nomination.get("pr_auc"),"stage8_holdout":final_metrics["pr_auc"]},
    {"metric":"roc_auc","stage6_oof":stage6_nomination.get("roc_auc"),"stage8_holdout":final_metrics["roc_auc"]},
    {"metric":"brier_score","stage6_oof":stage6_nomination.get("brier_score"),"stage8_holdout":final_metrics["brier_score"]},
])

comparison["absolute_change"] = (
    comparison["stage8_holdout"] - comparison["stage6_oof"]
)

display(comparison.round(4))

comparison.to_csv(
    METRICS_DIR/"stage6_vs_stage8_comparison.csv",
    index=False
)

,metric,stage6_oof,stage8_holdout,absolute_change
0,recall,0.8027,0.7931,-0.0096
1,precision,0.1958,0.1921,-0.0037
2,f1,0.3148,0.3092,-0.0056
3,pr_auc,0.4133,0.4198,0.0065
4,roc_auc,0.8868,0.8847,-0.0021
5,brier_score,0.1476,0.1491,0.0015


## 15. Final Stage 8 summary

In [17]:
holdout_triggered_dimensions = holdout_fairness_summary[
    holdout_fairness_summary["governance_trigger"]=="INVESTIGATE"
]["dimension"].tolist()

stage8_summary = {
    "project":PROJECT_NAME,
    "model":FROZEN_MODEL_NAME,
    "frozen_threshold":FROZEN_THRESHOLD,
    "holdout_used":True,
    "holdout_usage":"one-time final evaluation only",
    "model_retrained_on_holdout":False,
    "threshold_changed_after_holdout":False,
    "holdout_metrics":final_metrics,
    "holdout_fairness_triggered_dimensions":holdout_triggered_dimensions,
    "stage7_fairness_triggered_dimensions":
        stage7_governance.get("fairness_triggered_dimensions",[]),
    "historical_target_disclaimer":
        "HadHeartAttack is historical/self-reported; no future-event claim.",
    "stage8_status":
        "HOLDOUT EVALUATION COMPLETE - FINAL GOVERNANCE REVIEW REQUIRED",
    "next_stage":
        "Stage 9 final governance, model card and deployment-readiness decision"
}

summary_file = GOVERNANCE_DIR/"stage8_final_holdout_summary.json"
summary_file.write_text(
    json.dumps(stage8_summary,indent=2),
    encoding="utf-8"
)

print(json.dumps(stage8_summary,indent=2))

{
  "project": "Heart_Attack_Risk_Assessment",
  "model": "xgboost",
  "frozen_threshold": 0.52,
  "holdout_used": true,
  "holdout_usage": "one-time final evaluation only",
  "model_retrained_on_holdout": false,
  "threshold_changed_after_holdout": false,
  "holdout_metrics": {
    "accuracy": 0.798730970208338,
    "recall": 0.7931103146156909,
    "precision": 0.19205361878586238,
    "f1": 0.3092271262761539,
    "pr_auc": 0.4198124691659367,
    "roc_auc": 0.8846650440765925,
    "brier_score": 0.1491195112466812,
    "specificity": 0.799069455103607,
    "fpr": 0.20093054489639295,
    "fnr": 0.20688968538430905,
    "tn": 66636,
    "fp": 16756,
    "fn": 1039,
    "tp": 3983,
    "threshold": 0.52,
    "holdout_rows": 88414,
    "positive_prevalence": 0.056800959124120615
  },
  "holdout_fairness_triggered_dimensions": [
    "age",
    "sex",
    "race"
  ],
  "stage7_fairness_triggered_dimensions": [
    "age",
    "sex"
  ],
  "historical_target_disclaimer": "HadHeartAttack i

## 16. Compact final report

In [18]:
final_report = pd.DataFrame([
    {"item":"Model","value":FROZEN_MODEL_NAME},
    {"item":"Frozen threshold","value":FROZEN_THRESHOLD},
    {"item":"Recall","value":final_metrics["recall"]},
    {"item":"Precision","value":final_metrics["precision"]},
    {"item":"F1","value":final_metrics["f1"]},
    {"item":"PR-AUC","value":final_metrics["pr_auc"]},
    {"item":"ROC-AUC","value":final_metrics["roc_auc"]},
    {"item":"Brier Score","value":final_metrics["brier_score"]},
    {"item":"False Negatives","value":final_metrics["fn"]},
    {"item":"False Positives","value":final_metrics["fp"]},
    {
        "item":"Fairness triggers",
        "value":",".join(holdout_triggered_dimensions)
                if holdout_triggered_dimensions else "None"
    },
])

display(final_report)

final_report.to_csv(
    GOVERNANCE_DIR/"stage8_final_report.csv",
    index=False
)

,item,value
0,Model,xgboost
1,Frozen threshold,0.52
2,Recall,0.79311
3,Precision,0.192054
4,F1,0.309227
5,PR-AUC,0.419812
6,ROC-AUC,0.884665
7,Brier Score,0.14912
8,False Negatives,1039
9,False Positives,16756


In [20]:
display(
    holdout_fairness_summary.round(4)
)

,dimension,recall_gap,trigger_threshold,governance_trigger
0,age,0.6391,0.1,INVESTIGATE
1,sex,0.1169,0.1,INVESTIGATE
2,race,0.1668,0.1,INVESTIGATE


In [21]:
display(
    holdout_fairness_tables["race"].round(4)
)

,group,n,positive_actual_n,recall,precision,fnr,fpr,tn,fp,fn,tp
0,"Black only, Non-Hispanic",6879,314,0.7611,0.1566,0.2389,0.1960,5278,1287,75,239
1,Hispanic,8554,342,0.6725,0.1765,0.3275,0.1307,7139,1073,112,230
2,"Multiracial, Non-Hispanic",1912,112,0.8393,0.2260,0.1607,0.1789,1478,322,18,94
3,"Other race only, Non-Hispanic",4453,181,0.8343,0.1953,0.1657,0.1456,3650,622,30,151
4,"White only, Non-Hispanic",63912,3887,0.8032,0.1949,0.1968,0.2148,47132,12893,765,3122
5,nan,2704,186,0.7903,0.2082,0.2097,0.2220,1959,559,39,147


## 17. Log Stage 8 to MLflow

In [19]:
with mlflow.start_run(
    run_name=f"{TEAM_ID}_{STUDENT_ID}_stage8_locked_holdout_final"
) as run:

    mlflow.set_tags({
        "team_id":TEAM_ID,
        "student_id":STUDENT_ID,
        "project_name":PROJECT_NAME,
        "stage":"stage8_locked_holdout",
        "model":FROZEN_MODEL_NAME,
        "evaluation_type":"one_time_locked_holdout",
    })

    mlflow.log_params({
        "frozen_threshold":FROZEN_THRESHOLD,
        "holdout_rows":len(y_holdout),
        "holdout_used":True,
        "model_retrained_on_holdout":False,
        "threshold_changed_after_holdout":False,
        "bootstrap_repetitions":BOOTSTRAP_REPETITIONS,
    })

    mlflow.log_metrics({
        "holdout_accuracy":final_metrics["accuracy"],
        "holdout_recall":final_metrics["recall"],
        "holdout_precision":final_metrics["precision"],
        "holdout_f1":final_metrics["f1"],
        "holdout_pr_auc":final_metrics["pr_auc"],
        "holdout_roc_auc":final_metrics["roc_auc"],
        "holdout_brier_score":final_metrics["brier_score"],
        "holdout_specificity":final_metrics["specificity"],
        "holdout_false_negatives":float(final_metrics["fn"]),
        "holdout_false_positives":float(final_metrics["fp"]),
        "holdout_fairness_dimensions_triggered":
            float(len(holdout_triggered_dimensions)),
    })

    for folder in [METRICS_DIR,FAIRNESS_DIR,GOVERNANCE_DIR,PREDICTION_DIR]:
        mlflow.log_artifacts(
            str(folder),
            artifact_path=f"stage8/{folder.name}"
        )

    print("Stage 8 MLflow Run ID:", run.info.run_id)

Stage 8 MLflow Run ID: ec12625db7bb406180747a5eca9dac9a
🏃 View run team05_s502_stage8_locked_holdout_final at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/ec12625db7bb406180747a5eca9dac9a
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1


# Stage 8 completion criteria

Stage 8 is complete when:
- frozen XGBoost and threshold are reused;
- locked holdout is opened once;
- no tuning/retraining/threshold search occurs;
- final performance metrics are saved;
- confidence intervals are generated;
- age/sex/race fairness is rechecked;
- same 0.10 recall-gap trigger is applied;
- Stage 6 vs Stage 8 is compared;
- final evidence is logged to MLflow.

After completion, send the final table from Section 16 or upload:
`artifacts/stage8/governance/stage8_final_report.csv`

Then Stage 9 can be created using the actual final holdout evidence.